# Notebook 01 — Fine-Tuning BERT for Text Classification

In this notebook we fine-tune a pre-trained BERT model on the IMDB sentiment dataset using HuggingFace Transformers.

In [ ]:
# Install dependencies (run once)
# !pip install transformers datasets accelerate evaluate scikit-learn

## 1. Load the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)
print(dataset["train"][0])

## 2. Tokenise

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Tokenised splits:", tokenized)

## 3. Define the model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(model.config)

## 4. Train

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results_bert",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"].select(range(2000)),  # small subset for demo
    eval_dataset=tokenized["test"].select(range(500)),
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Evaluate and infer

In [ ]:
results = trainer.evaluate()
print("Eval results:", results)

# Single-sample inference
from transformers import pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)
print(classifier("This movie was absolutely fantastic!"))
print(classifier("Terrible film, complete waste of time."))